# Model Comparison: YOLO26m vs Faster R-CNN

This notebook compares the performance of two face mask detection models:
- **YOLO26m**: Object detection model (`face_mask_detection_yolo26m_v1_best.pt`)
- **Faster R-CNN**: End-to-end detection + classification (`face_mask_detection_faster_rcnn_final.pt`)


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score,
)
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


In [ ]:
# Define paths
BASE_DIR = Path('/home/khoanguyen/workspace/UIT/face_mask_detection')
YOLO26M_MODEL_PATH = BASE_DIR / 'models' / 'face_mask_detection_yolo26m_v1_best.pt'
FRCNN_MODEL_PATH = BASE_DIR / 'models' / 'face_mask_detection_faster_rcnn_final.pt'
TEST_IMAGES_DIR = BASE_DIR / 'datasets' / 'face-mask-detection-processed' / 'images' / 'test'
TEST_LABELS_DIR = BASE_DIR / 'datasets' / 'face-mask-detection-processed' / 'labels' / 'test'

CLASS_NAMES = ['With Mask', 'Without Mask', 'Mask Weared Incorrect']
# Faster R-CNN uses 1-indexed labels (0 = background)
FRCNN_CLASS_NAMES = {1: 'With Mask', 2: 'Without Mask', 3: 'Mask Weared Incorrect'}

print(f"YOLO26m model exists:      {YOLO26M_MODEL_PATH.exists()}")
print(f"Faster R-CNN model exists: {FRCNN_MODEL_PATH.exists()}")
print(f"Test images dir exists:    {TEST_IMAGES_DIR.exists()}")
print(f"Test labels dir exists:    {TEST_LABELS_DIR.exists()}")


# Faster R-CNN helpers
def create_frcnn_model(num_classes=4):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def load_frcnn_model(model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    model = create_frcnn_model(num_classes=4)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  Loaded checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
        print(f"  Val loss: {checkpoint.get('val_loss', 'unknown')}")
    else:
        model.load_state_dict(checkpoint)
    model.to(device)
    model.eval()
    return model


def frcnn_predict(model, image_rgb, device, score_threshold=0.5):
    img_tensor = torch.as_tensor(image_rgb, dtype=torch.float32).permute(2, 0, 1) / 255.0
    with torch.no_grad():
        output = model([img_tensor.to(device)])[0]
    boxes = output['boxes'].cpu().numpy()
    labels = output['labels'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    keep = scores >= score_threshold
    return boxes[keep], labels[keep], scores[keep]


In [ ]:
# Load models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("\nLoading YOLO26m model...")
yolo_model = YOLO(str(YOLO26M_MODEL_PATH))

print("Loading Faster R-CNN model...")
frcnn_model = load_frcnn_model(FRCNN_MODEL_PATH, device)

print("\nBoth models loaded successfully!")


In [ ]:
# Get test images
test_images = sorted(
    list(TEST_IMAGES_DIR.glob('*.jpg')) + list(TEST_IMAGES_DIR.glob('*.png'))
)
print(f"Found {len(test_images)} test images")


# ── Shared helper functions ──────────────────────────────────────────────────
def read_yolo_label(label_path):
    """Read YOLO-format label → list of (class_id, cx, cy, bw, bh)."""
    labels = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    labels.append((int(parts[0]), *[float(x) for x in parts[1:]]))
    return labels


def yolo_to_pixel(cx, cy, bw, bh, img_w, img_h):
    x1 = max(0, int((cx - bw / 2) * img_w))
    y1 = max(0, int((cy - bh / 2) * img_h))
    x2 = min(img_w, int((cx + bw / 2) * img_w))
    y2 = min(img_h, int((cy + bh / 2) * img_h))
    return x1, y1, x2, y2


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    if x2 < x1 or y2 < y1:
        return 0.0
    inter = (x2 - x1) * (y2 - y1)
    union = ((box1[2]-box1[0])*(box1[3]-box1[1]) +
             (box2[2]-box2[0])*(box2[3]-box2[1]) - inter)
    return inter / union if union > 0 else 0.0


print("Sample test images:")
for p in test_images[:3]:
    print(f"  {p.name}")


## Faster R-CNN Model Evaluation

End-to-end detection + classification pipeline using Faster R-CNN.
IoU-based matching (threshold = 0.5) against ground truth; unmatched predictions/GT are treated as false positives/negatives respectively.


In [ ]:
# Faster R-CNN evaluation (IoU-based matching)
SCORE_THR = 0.5
IOU_THR   = 0.5

frcnn_y_true = []
frcnn_y_pred = []
frcnn_confidences = []
frcnn_false_positives = 0

for img_path in tqdm(test_images, desc='Faster R-CNN'):
    label_path = TEST_LABELS_DIR / (img_path.stem + '.txt')
    gt_labels  = read_yolo_label(label_path)

    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_h, img_w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    pred_boxes, pred_labels, pred_scores = frcnn_predict(
        frcnn_model, img_rgb, device, score_threshold=SCORE_THR
    )

    # Convert GT to pixel coords (1-indexed labels for FRCNN)
    gt_pixel = []
    for (gt_cls, cx, cy, bw, bh) in gt_labels:
        x1, y1, x2, y2 = yolo_to_pixel(cx, cy, bw, bh, img_w, img_h)
        gt_pixel.append((gt_cls + 1, x1, y1, x2, y2))  # shift to 1-indexed

    matched_gt = set()
    for p_idx, (p_box, p_cls, p_score) in enumerate(zip(pred_boxes, pred_labels, pred_scores)):
        px1, py1, px2, py2 = map(int, p_box)
        best_iou = IOU_THR
        best_gt_idx = -1
        best_gt_cls = None

        for g_idx, (gt_cls1, gx1, gy1, gx2, gy2) in enumerate(gt_pixel):
            if g_idx in matched_gt:
                continue
            iou = calculate_iou((px1, py1, px2, py2), (gx1, gy1, gx2, gy2))
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g_idx
                best_gt_cls = gt_cls1

        if best_gt_cls is not None:
            matched_gt.add(best_gt_idx)
            # Convert back to 0-indexed to align with CLASS_NAMES
            frcnn_y_true.append(best_gt_cls - 1)
            frcnn_y_pred.append(int(p_cls) - 1)
            frcnn_confidences.append(float(p_score))
        else:
            frcnn_false_positives += 1

print(f"\nFaster R-CNN evaluation complete!")
print(f"  Matched detections: {len(frcnn_y_pred)}")
print(f"  False positives (no GT match): {frcnn_false_positives}")


In [ ]:
# Faster R-CNN metrics
frcnn_accuracy  = accuracy_score(frcnn_y_true, frcnn_y_pred)
frcnn_precision = precision_score(frcnn_y_true, frcnn_y_pred, average='weighted', zero_division=0)
frcnn_recall    = recall_score(frcnn_y_true, frcnn_y_pred, average='weighted', zero_division=0)
frcnn_f1        = f1_score(frcnn_y_true, frcnn_y_pred, average='weighted', zero_division=0)

frcnn_precision_per_class = precision_score(frcnn_y_true, frcnn_y_pred, average=None, zero_division=0)
frcnn_recall_per_class    = recall_score(frcnn_y_true, frcnn_y_pred, average=None, zero_division=0)
frcnn_f1_per_class        = f1_score(frcnn_y_true, frcnn_y_pred, average=None, zero_division=0)

print("Faster R-CNN Model Metrics (on matched detections):")
print(f"  Accuracy:       {frcnn_accuracy:.4f}")
print(f"  Precision:      {frcnn_precision:.4f}")
print(f"  Recall:         {frcnn_recall:.4f}")
print(f"  F1-Score:       {frcnn_f1:.4f}")
print(f"  Avg Confidence: {np.mean(frcnn_confidences):.4f}")
print(f"\n  Detection Statistics:")
print(f"    Matched detections: {len(frcnn_y_pred)}")
print(f"    False positives:    {frcnn_false_positives}")
print("\nPer-class metrics:")
for i, name in enumerate(CLASS_NAMES):
    if i < len(frcnn_precision_per_class):
        print(f"  {name}:")
        print(f"    Precision: {frcnn_precision_per_class[i]:.4f}")
        print(f"    Recall:    {frcnn_recall_per_class[i]:.4f}")
        print(f"    F1-Score:  {frcnn_f1_per_class[i]:.4f}")


## YOLO26m Model Evaluation

Detection + classification evaluation using YOLO26m.
IoU-based matching (threshold = 0.5) is used against ground truth labels.


In [ ]:
# YOLO26m evaluation (IoU-based matching)
yolo_y_true = []
yolo_y_pred = []
yolo_confidences = []
yolo_false_positives = 0

for img_path in tqdm(test_images, desc='YOLO26m'):
    label_path = TEST_LABELS_DIR / (img_path.stem + '.txt')
    gt_labels = read_yolo_label(label_path)

    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_h, img_w = img.shape[:2]

    # Run YOLO26m prediction
    results = yolo_model(str(img_path), verbose=False)

    pred_boxes = []
    for result in results:
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        for box in boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, xyxy)
            pred_class = int(box.cls[0])
            confidence = float(box.conf[0])
            pred_boxes.append((x1, y1, x2, y2, pred_class, confidence))

    matched_gt = set()
    for pred_box in pred_boxes:
        px1, py1, px2, py2, pred_class, confidence = pred_box

        best_iou = IOU_THR
        best_gt_idx = -1
        best_gt_class = None

        for gt_idx, (gt_class, cx, cy, bw, bh) in enumerate(gt_labels):
            if gt_idx in matched_gt:
                continue
            gx1, gy1, gx2, gy2 = yolo_to_pixel(cx, cy, bw, bh, img_w, img_h)
            iou = calculate_iou((px1, py1, px2, py2), (gx1, gy1, gx2, gy2))
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
                best_gt_class = gt_class

        if best_gt_class is not None:
            matched_gt.add(best_gt_idx)
            yolo_y_true.append(best_gt_class)
            yolo_y_pred.append(pred_class)
            yolo_confidences.append(confidence)
        else:
            yolo_false_positives += 1

print(f"\nYOLO26m evaluation complete!")
print(f"  Matched detections: {len(yolo_y_pred)}")
print(f"  False positives (no GT match): {yolo_false_positives}")


In [ ]:
# YOLO26m metrics
yolo_accuracy = accuracy_score(yolo_y_true, yolo_y_pred)
yolo_precision = precision_score(yolo_y_true, yolo_y_pred, average='weighted', zero_division=0)
yolo_recall = recall_score(yolo_y_true, yolo_y_pred, average='weighted', zero_division=0)
yolo_f1 = f1_score(yolo_y_true, yolo_y_pred, average='weighted', zero_division=0)

yolo_precision_per_class = precision_score(yolo_y_true, yolo_y_pred, average=None, zero_division=0)
yolo_recall_per_class = recall_score(yolo_y_true, yolo_y_pred, average=None, zero_division=0)
yolo_f1_per_class = f1_score(yolo_y_true, yolo_y_pred, average=None, zero_division=0)

print("YOLO26m Model Metrics (on matched detections):")
print(f"  Accuracy:       {yolo_accuracy:.4f}")
print(f"  Precision:      {yolo_precision:.4f}")
print(f"  Recall:         {yolo_recall:.4f}")
print(f"  F1-Score:       {yolo_f1:.4f}")
print(f"  Avg Confidence: {np.mean(yolo_confidences):.4f}")
print(f"\n  Detection Statistics:")
print(f"    Matched detections: {len(yolo_y_pred)}")
print(f"    False positives:    {yolo_false_positives}")
print("\nPer-class metrics:")
for i, name in enumerate(CLASS_NAMES):
    if i < len(yolo_precision_per_class):
        print(f"  {name}:")
        print(f"    Precision: {yolo_precision_per_class[i]:.4f}")
        print(f"    Recall:    {yolo_recall_per_class[i]:.4f}")
        print(f"    F1-Score:  {yolo_f1_per_class[i]:.4f}")


## Model Comparison: YOLO26m vs Faster R-CNN

**Fair Classification Comparison:**
- Both models are evaluated using **IoU-based matching** (threshold = 0.5) against ground truth.
- All metrics are computed on **matched detections only**.
- Detection statistics (false positives) are reported separately.


In [ ]:
# Comparison table
comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Avg Confidence'],
    'YOLO26m': [
        f'{yolo_accuracy:.4f}',
        f'{yolo_precision:.4f}',
        f'{yolo_recall:.4f}',
        f'{yolo_f1:.4f}',
        f'{np.mean(yolo_confidences):.4f}',
    ],
    'Faster R-CNN': [
        f'{frcnn_accuracy:.4f}',
        f'{frcnn_precision:.4f}',
        f'{frcnn_recall:.4f}',
        f'{frcnn_f1:.4f}',
        f'{np.mean(frcnn_confidences):.4f}',
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("OVERALL MODEL COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)


In [ ]:
# Per-class comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_list = ['Precision', 'Recall', 'F1-Score']
yolo_vals = [yolo_precision_per_class, yolo_recall_per_class, yolo_f1_per_class]
frcnn_vals = [frcnn_precision_per_class, frcnn_recall_per_class, frcnn_f1_per_class]

for idx, (metric_name, y_vals, rc_vals) in enumerate(zip(metrics_list, yolo_vals, frcnn_vals)):
    ax = axes[idx]
    x = np.arange(len(CLASS_NAMES))
    width = 0.35

    yolo_plot = np.zeros(len(CLASS_NAMES))
    frcnn_plot = np.zeros(len(CLASS_NAMES))
    for i in range(len(CLASS_NAMES)):
        if i < len(y_vals):
            yolo_plot[i] = y_vals[i]
        if i < len(rc_vals):
            frcnn_plot[i] = rc_vals[i]

    ax.bar(x - width / 2, yolo_plot, width, label='YOLO26m', alpha=0.8, color='seagreen')
    ax.bar(x + width / 2, frcnn_plot, width, label='Faster R-CNN', alpha=0.8, color='steelblue')

    ax.set_xlabel('Class')
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} by Class')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

yolo_cm = confusion_matrix(yolo_y_true, yolo_y_pred, labels=[0, 1, 2])
frcnn_cm = confusion_matrix(frcnn_y_true, frcnn_y_pred, labels=[0, 1, 2])

sns.heatmap(yolo_cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('YOLO26m Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(frcnn_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title('Faster R-CNN Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()


In [ ]:
# Overall metrics comparison bar chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
yolo_overall = [yolo_accuracy, yolo_precision, yolo_recall, yolo_f1]
frcnn_overall = [frcnn_accuracy, frcnn_precision, frcnn_recall, frcnn_f1]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width / 2, yolo_overall, width, label='YOLO26m', alpha=0.8, color='seagreen')
bars2 = ax.bar(x + width / 2, frcnn_overall, width, label='Faster R-CNN', alpha=0.8, color='steelblue')

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison on Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Detailed Classification Reports
print("=" * 80)
print("YOLO26M CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(
    yolo_y_true, yolo_y_pred,
    labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0
))

print("\n" + "=" * 80)
print("FASTER R-CNN CLASSIFICATION REPORT")
print("=" * 80)
print(classification_report(
    frcnn_y_true, frcnn_y_pred,
    labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0
))
